# AAMI ECG Heartbeat Classification with Classical Machine Learning

**Project walkthrough and reproducibility notebook**  
Dataset: MIT-BIH Arrhythmia Database · Models: Logistic Regression and Random Forest

## Abstract

This project reads ECG recordings and expert annotations, extracts heartbeat-centered signal windows, calculates numerical features, groups the data by patient, and evaluates classical machine-learning models. The purpose of this notebook is to explain the steps and make them reproducible.

> **Important evaluation note:** the processed dataset contains very few Q beats (15 in the current feature table; only 7 in the DS2 test set). Therefore, the five-class results for Q are not statistically reliable. Report this limitation clearly. Do not tune the final model using DS2.

## Roadmap

1. Understand the files and labels.
2. Read ECG and annotation files with WFDB.
3. Extract beat-centered windows and features.
4. Check the data and patient-independent split.
5. Fit models with grouped cross-validation.
6. Evaluate once on DS2 and discuss limitations.

## 1. How the data flows

```text
.dat signal + .hea header + .atr annotations
                    │ WFDB reads them together
                    ▼
      ECG segment around each annotated beat
                    │ calculate RR and amplitude features
                    ▼
        one row per heartbeat in features.csv
                    │ group by patient: DS1 train, DS2 test
                    ▼
      Logistic Regression / Random Forest evaluation
```

A “window” is a short slice of signal around an annotated beat. This project takes 0.20 seconds before and 0.40 seconds after the annotation. The current feature set is offline because it includes the time to the following beat.

## 2. What are the MIT-BIH files?

A record is a group of files sharing a name, such as `100`:

| File | What it contains | Used by this pipeline? |
|---|---|---|
| `100.dat` | Encoded ECG signal samples | Yes |
| `100.hea` | Signal metadata: channel names, sampling rate, scaling | Yes, WFDB uses it to decode the signal |
| `100.atr` | Annotation positions and beat symbols | Yes |
| `100.xws` | Display settings for waveform viewers | No |

WFDB is an external Python package for reading waveform databases. It is not written by this project. The `mitdb/mitdbdir/` directory is documentation accompanying the database; its old C source files generate documentation tables, not the WFDB library used here.

## 3. Setup and paths

Run this notebook from the project root, where `mitdb/`, `config/`, and `data/` are located. Install project dependencies first if needed:

```bash
python -m pip install -r requirements.txt
```

The project already has local MIT-BIH files, so downloading is unnecessary.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
DATA_DIR = ROOT / "mitdb"
PROTOCOL_PATH = ROOT / "config" / "protocol.json"
FEATURE_PATH = ROOT / "data" / "processed" / "features.csv"
OUTPUT_DIR = ROOT / "outputs" / "smoke"

print("Project root:", ROOT)
print("Local MIT-BIH folder exists:", DATA_DIR.exists())
print("Processed feature CSV exists:", FEATURE_PATH.exists())

## 4. Protocol: labels and patient split

MIT-BIH contains detailed annotation symbols. The protocol groups supported symbols into five AAMI classes. Unrecognized symbols cause preprocessing to stop rather than being silently guessed. The exact mapping and record lists live in `config/protocol.json`.

In [ ]:
protocol = json.loads(PROTOCOL_PATH.read_text())
print(protocol["protocol"])
print("
Annotation-to-class mapping:")
for class_name, symbols in protocol["label_mapping"].items():
    print(f"  {class_name}: {symbols}")
print("
Excluded symbols:", protocol["excluded_symbols"])
print("
DS1 train records:", protocol["ds1"])
print("DS2 test records:", protocol["ds2"])
print("
Protocol note:", protocol["split_deviation"])

### Class meanings

- **N:** normal and selected related beats
- **S:** supraventricular ectopic beats
- **V:** ventricular ectopic beats
- **F:** fusion beats
- **Q:** paced or unclassifiable beats, according to this configured mapping

Records 201 and 202 are from the same person. This project keeps both in DS1 so that the same patient cannot appear in both train and test partitions. The commonly reproduced split is adjusted for that reason.

## 5. Inspect one raw record with WFDB

WFDB reads the header, signal, and annotations as related parts of one record. The example below displays a short part of record 100 and several annotation locations. Annotation symbols are original expert codes, not yet the final AAMI labels.

In [ ]:
import wfdb

record_name = "100"
record_path = DATA_DIR / record_name
record = wfdb.rdrecord(str(record_path), physical=True)
annotation = wfdb.rdann(str(record_path), "atr")

print("Sampling frequency:", record.fs, "Hz")
print("Signal channels:", record.sig_name)
print("Signal array shape:", record.p_signal.shape)
print("First annotations:")
for sample, symbol in zip(annotation.sample[:10], annotation.symbol[:10]):
    print(f"  sample={sample:>7}, time={sample / record.fs:>7.2f} s, symbol={symbol}")

In [ ]:
# Plot a short segment of the first available signal channel.
seconds_to_show = 5
n = int(seconds_to_show * record.fs)
t = np.arange(n) / record.fs
plt.figure(figsize=(12, 3))
plt.plot(t, record.p_signal[:n, 0], linewidth=0.8)
plt.xlabel("Time (seconds)")
plt.ylabel("Signal (mV)")
plt.title(f"MIT-BIH record {record_name}, channel {record.sig_name[0]}")
plt.tight_layout()
plt.show()

## 6. Convert annotations into heartbeat examples

For each recognized beat annotation, the preprocessing code:

1. Chooses MLII when available (otherwise the first channel, unless a lead is specified).
2. Converts 0.20 seconds and 0.40 seconds into sample counts using the record's sampling frequency.
3. Extracts that signal window centered on the annotated location.
4. Calculates RR interval and amplitude features and attaches the AAMI target.
5. Keeps identifiers and original annotation symbols for audit, while excluding them from model inputs.

The next cell includes the project's preprocessing implementation. Running feature extraction over every record takes a little while; by default this notebook reads the already-built CSV. Change `REBUILD_FEATURES` to `True` only when you want to regenerate it.

In [ ]:
# Project preprocessing implementation (same functions as preprocessing.py).
"""MIT-BIH ingestion and transparent AAMI feature extraction (offline protocol)."""
from __future__ import annotations
import argparse, json
from pathlib import Path
import numpy as np
import pandas as pd

CLASSES = ('N','S','V','F','Q')

def load_protocol(path='config/protocol.json'):
    return json.loads(Path(path).read_text())

def patient_id(record: str) -> str:
    # MIT-BIH records 201 and 202 are the two recordings of one subject.
    return '201_202' if str(record) in {'201','202'} else str(record)

def validate_dataset(root, records=None):
    root=Path(root)
    records=records or (root/'RECORDS').read_text().split()
    missing=[]
    for rec in records:
        for suffix in ('.hea','.dat','.atr'):
            if not (root/f'{rec}{suffix}').exists(): missing.append(f'{rec}{suffix}')
    if missing: raise FileNotFoundError('Missing required MIT-BIH files: '+', '.join(missing[:20]))
    return records

def _label(symbol, mapping, excluded):
    for cls, symbols in mapping.items():
        if symbol in symbols: return cls
    if symbol in excluded: return None
    raise ValueError(f'Unexpected annotation symbol {symbol!r}; add it explicitly to mapping or exclusions')

def extract_record(root, rec, protocol, lead=None):
    try:
        import wfdb
    except ImportError as exc:
        raise RuntimeError("WFDB is required for ECG extraction; install requirements.txt") from exc
    path=str(Path(root)/rec)
    record=wfdb.rdrecord(path, physical=True)
    ann=wfdb.rdann(path, 'atr')
    names=list(record.sig_name)
    if lead is None:
        lead_idx=next((i for i,n in enumerate(names) if n.strip().upper()=='MLII'),0)
    else:
        if lead not in names: raise ValueError(f'{rec}: lead {lead!r} not found; available {names}')
        lead_idx=names.index(lead)
    signal=np.asarray(record.p_signal[:,lead_idx],dtype=float)
    fs=float(record.fs)
    rpeaks=np.asarray(ann.sample,dtype=int)
    # Only beat annotations yield target labels. Preserve unknown-symbol failure.
    beat_items=[]
    for idx,(sample,symbol) in enumerate(zip(ann.sample,ann.symbol)):
        label=_label(symbol,protocol['label_mapping'],protocol['excluded_symbols'])
        if label is not None: beat_items.append((idx,int(sample),symbol,label))
    pre=int(round(protocol['window_seconds']['pre_r']*fs)); post=int(round(protocol['window_seconds']['post_r']*fs))
    rows=[]
    for position,(ann_idx,sample,symbol,label) in enumerate(beat_items):
        start,end=sample-pre,sample+post
        if start<0 or end>len(signal): continue
        win=signal[start:end]
        if len(win)!=pre+post or not np.isfinite(win).all(): continue
        prev=(sample-beat_items[position-1][1])/fs if position>0 else np.nan
        nxt=(beat_items[position+1][1]-sample)/fs if position+1<len(beat_items) else np.nan
        # Local morphology is descriptive, not delineation. P/Q/S/T and intervals remain NA.
        r=float(signal[sample]); baseline=float(np.median(win[:max(1,pre//2)]))
        qrs=win[max(0,pre-int(.08*fs)):min(len(win),pre+int(.10*fs))]
        rows.append({
          'record_id':str(rec),'patient_id':patient_id(str(rec)),'annotation_symbol':symbol,
          'sample_index':sample,'sampling_hz':fs,'target':label,
          'rr_previous_s':prev,'rr_following_s':nxt,'rr_ratio_next_prev':nxt/prev if np.isfinite(nxt) and np.isfinite(prev) and prev else np.nan,
          'rr_local_mean_2_s':float(np.nanmean([x for x in [prev,nxt] if np.isfinite(x)])) if np.isfinite(prev) or np.isfinite(nxt) else np.nan,
          'r_amplitude_mV':r,'r_amplitude_relative_mV':r-baseline,
          'qrs_window_peak_to_peak_mV':float(np.ptp(qrs)) if len(qrs) else np.nan,
          'p_amplitude_mV':np.nan,'q_amplitude_mV':np.nan,'s_amplitude_mV':np.nan,'t_amplitude_mV':np.nan,
          'qrs_duration_s':np.nan,'pr_interval_s':np.nan,'qt_interval_s':np.nan,
          'window_start_sample':start,'window_end_sample':end,'window_valid':True,'_window':win.astype(np.float32)})
    return rows

def build_features(root, output, protocol_path='config/protocol.json', lead=None):
    p=load_protocol(protocol_path); records=validate_dataset(root)
    valid=set(p['ds1']+p['ds2'])
    records=[r for r in records if r in valid]
    rows=[]
    for i,rec in enumerate(records,1):
        print(f'[{i}/{len(records)}] {rec}',flush=True)
        rows.extend(extract_record(root,rec,p,lead))
    if not rows: raise ValueError('No labeled heartbeats extracted')
    windows=np.stack([r.pop('_window') for r in rows])
    df=pd.DataFrame(rows)
    if df.empty: raise ValueError('No labeled heartbeats extracted')
    unexpected=set(df.target)-set(CLASSES)
    if unexpected: raise ValueError(f'Invalid labels: {unexpected}')
    if df.duplicated(['record_id','sample_index']).any(): raise ValueError('Duplicate beat keys')
    out=Path(output);out.parent.mkdir(parents=True,exist_ok=True);df.to_csv(out,index=False)
    np.savez_compressed(out.with_name('heartbeat_windows.npz'), windows=windows, record_id=df.record_id.astype(str).to_numpy(), sample_index=df.sample_index.to_numpy(), sampling_hz=df.sampling_hz.to_numpy())
    schema={'target':'target','group':'patient_id','metadata':['record_id','patient_id','annotation_symbol','sample_index','sampling_hz','window_start_sample','window_end_sample','window_valid'],
            'model_features':[c for c in df.columns if c not in {'target','record_id','patient_id','annotation_symbol','sample_index','sampling_hz','window_start_sample','window_end_sample','window_valid'}],
            'classes':list(CLASSES),'window_archive':'heartbeat_windows.npz (float32, rows aligned to feature CSV)','rows':len(df),'class_counts':df.target.value_counts().reindex(CLASSES,fill_value=0).to_dict()}
    out.with_suffix('.schema.json').write_text(json.dumps(schema,indent=2))
    dictionary={
      'rr_previous_s':{'definition':'Seconds between this beat and the previous mapped beat','unit':'s','time':'past'},
      'rr_following_s':{'definition':'Seconds between this beat and the next mapped beat','unit':'s','time':'future; offline only'},
      'rr_ratio_next_prev':{'definition':'Following RR divided by previous RR','unit':'ratio','time':'past and future'},
      'rr_local_mean_2_s':{'definition':'Mean of available previous and following mapped-beat RR intervals (at most two)','unit':'s','time':'past and future'},
      'r_amplitude_mV':{'definition':'Signal at annotated beat sample','unit':'mV','time':'current sample'},
      'r_amplitude_relative_mV':{'definition':'Annotated sample amplitude minus median of first half of pre-R window','unit':'mV','time':'past/current'},
      'qrs_window_peak_to_peak_mV':{'definition':'Peak-to-peak signal range in fixed 80 ms pre to 100 ms post annotation window; not delineated QRS duration','unit':'mV','time':'past and future'},
      'p_amplitude_mV':{'definition':'Unavailable; no validated P-wave delineation','unit':'mV','time':'not measured'},
      'q_amplitude_mV':{'definition':'Unavailable; no validated Q-wave delineation','unit':'mV','time':'not measured'},
      's_amplitude_mV':{'definition':'Unavailable; no validated S-wave delineation','unit':'mV','time':'not measured'},
      't_amplitude_mV':{'definition':'Unavailable; no validated T-wave delineation','unit':'mV','time':'not measured'},
      'qrs_duration_s':{'definition':'Unavailable; no validated QRS onset/offset delineation','unit':'s','time':'not measured'},
      'pr_interval_s':{'definition':'Unavailable; no validated P onset/QRS onset delineation','unit':'s','time':'not measured'},
      'qt_interval_s':{'definition':'Unavailable; no validated QRS onset/T offset delineation','unit':'s','time':'not measured'}
    }
    out.with_name('data_dictionary.json').write_text(json.dumps(dictionary,indent=2))
    return df

In [ ]:
REBUILD_FEATURES = False
if REBUILD_FEATURES:
    features = build_features(DATA_DIR, FEATURE_PATH, PROTOCOL_PATH)
else:
    if not FEATURE_PATH.exists():
        raise FileNotFoundError("Feature CSV is missing; set REBUILD_FEATURES=True after installing WFDB.")
    features = pd.read_csv(FEATURE_PATH)

print("Feature table shape:", features.shape)
display(features.head())

## 7. Understand the processed table

Each row represents one heartbeat. `target` is the answer the model learns to predict. Fields such as `patient_id`, `record_id`, `annotation_symbol`, and `sample_index` are kept to trace the row to its source, but they are not predictor features.

Several morphology columns (P/Q/S/T amplitudes and PR/QT/QRS intervals) are deliberately missing: this project does not implement reliable wave delineation, so it does not invent those measurements.

In [ ]:
print("Rows per AAMI class:")
display(features["target"].value_counts().reindex(["N", "S", "V", "F", "Q"], fill_value=0).rename("beats"))

feature_cols = [
    c for c in features.select_dtypes(include="number").columns
    if c not in {
        "sample_index", "sampling_hz", "window_start_sample", "window_end_sample"
    }
]
missing_fraction = features[feature_cols].isna().mean().sort_values(ascending=False)
print("
Predictor missing-value fractions:")
display(missing_fraction.to_frame("missing_fraction"))

## 8. Patient-independent DS1/DS2 split

DS1 is used for model selection. DS2 is kept aside for final evaluation. Grouped cross-validation also keeps each patient together inside DS1. This is important because beats from one person can be similar; placing the same patient on both sides can make results look better than performance on unseen patients.

In [ ]:
ds1_records = set(protocol["ds1"])
ds2_records = set(protocol["ds2"])
train_df = features[features.record_id.astype(str).isin(ds1_records)].copy()
test_df = features[features.record_id.astype(str).isin(ds2_records)].copy()

train_patients = set(train_df.patient_id.astype(str))
test_patients = set(test_df.patient_id.astype(str))
train_record_ids = set(train_df.record_id.astype(str))
test_record_ids = set(test_df.record_id.astype(str))

assert not (train_patients & test_patients), "Patient overlap between train and test"
assert not (train_record_ids & test_record_ids), "Record overlap between train and test"
print(f"DS1: {len(train_df):,} beats from {len(train_patients)} patients")
print(f"DS2: {len(test_df):,} beats from {len(test_patients)} patients")
print("DS1 classes:", train_df.target.value_counts().to_dict())
print("DS2 classes:", test_df.target.value_counts().to_dict())

## 9. Model training

Both models use a scikit-learn `Pipeline`, so missing-value imputation (and scaling for Logistic Regression) is learned only from each training fold. Model selection uses macro-F1 on grouped DS1 folds. Macro-F1 gives each class equal importance, unlike accuracy, which can be dominated by the common N class.

The original command-line research search can use every CPU core and run many fits. To avoid a surprise long/hot run, this notebook does **not** retrain by default. Existing quick-run results are loaded below. Set `RUN_TRAINING=True` to train; use the quick configuration first and consider limiting CPU parallelism in `modeling.py` (`GridSearchCV(..., n_jobs=2)`).

In [ ]:
# Project modeling implementation (same functions as modeling.py, with search parallelism limited to 2 CPU jobs).
"""Patient-wise AAMI training; all learned transforms are fold-local pipelines."""
from __future__ import annotations
import argparse,json
from pathlib import Path
import joblib,numpy as np,pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report,confusion_matrix,f1_score
from sklearn.model_selection import GridSearchCV,StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

META={'record_id','patient_id','annotation_symbol','sample_index','sampling_hz','target','window_start_sample','window_end_sample','window_valid','split'}
def feature_columns(df,target='target'):
    if target not in df or 'patient_id' not in df or 'record_id' not in df: raise ValueError('CSV must include target, patient_id, record_id')
    cols=[c for c in df.select_dtypes(include='number').columns if c not in META]
    if not cols: raise ValueError('No numeric predictor features')
    if any(c.lower() in {'label','class','target','annotation','annotation_symbol'} for c in cols): raise ValueError('Label-like column entered as predictor')
    return cols

def check_partitions(train,test):
    p=set(train.patient_id.astype(str))&set(test.patient_id.astype(str))
    r=set(train.record_id.astype(str))&set(test.record_id.astype(str))
    if p: raise ValueError(f'Patient overlap across partitions: {sorted(p)}')
    if r: raise ValueError(f'Record overlap across partitions: {sorted(r)}')

def _models(seed,quick=False):
    n=[100] if quick else [200,400]
    return {
      'logistic_regression':(Pipeline([('imputer',SimpleImputer(strategy='median',keep_empty_features=True)),('scaler',StandardScaler()),('classifier',LogisticRegression(max_iter=3000,random_state=seed))]),{'classifier__C':[.1,1,10],'classifier__class_weight':[None,'balanced']}),
      'random_forest':(Pipeline([('imputer',SimpleImputer(strategy='median',keep_empty_features=True)),('classifier',RandomForestClassifier(random_state=seed,n_jobs=1))]),{'classifier__n_estimators':n,'classifier__max_depth':[None,20],'classifier__min_samples_split':[2,5],'classifier__min_samples_leaf':[1,3],'classifier__max_features':['sqrt',.8],'classifier__class_weight':[None,'balanced','balanced_subsample']})}

def run_training(csv,output,seed=42,quick=False):
    df=pd.read_csv(csv)
    if df.empty: raise ValueError('Feature dataset is empty')
    if not df.window_valid.astype(bool).all(): df=df[df.window_valid.astype(bool)].copy()
    if df.target.isna().any(): raise ValueError('Missing targets')
    numeric=df.select_dtypes(include='number').drop(columns=[c for c in META if c in df],errors='ignore')
    if np.isinf(numeric.to_numpy(dtype=float)).any(): raise ValueError('Infinite numeric values')
    if df.duplicated(['record_id','sample_index']).any(): raise ValueError('Duplicate heartbeat keys')
    p=json.loads(Path('config/protocol.json').read_text())
    ds1=set(p['ds1']); ds2=set(p['ds2'])
    train=df[df.record_id.astype(str).isin(ds1)].copy(); test=df[df.record_id.astype(str).isin(ds2)].copy()
    check_partitions(train,test)
    if train.empty or test.empty: raise ValueError('Expected DS1 and DS2 records are missing')
    features=feature_columns(train); features=[x for x in features if x in test]
    if not features: raise ValueError('No shared features')
    groups=train.patient_id.astype(str); y=train.target.astype(str)
    n_splits=min(3 if quick else 5,groups.nunique())
    if n_splits<2: raise ValueError('Need at least two DS1 patients for grouped CV')
    cv=StratifiedGroupKFold(n_splits=n_splits,shuffle=True,random_state=seed)
    splits=list(cv.split(train[features],y,groups))
    fold_class_counts=[]
    all_classes=sorted(set(y))
    for i,(itr,iva) in enumerate(splits):
        if set(groups.iloc[itr])&set(groups.iloc[iva]): raise ValueError(f'Patient overlap in CV fold {i}')
        if not len(set(y.iloc[iva])): raise ValueError('Empty validation fold')
        fold_class_counts.append({'fold':i,'train':y.iloc[itr].value_counts().reindex(all_classes,fill_value=0).to_dict(),'validation':y.iloc[iva].value_counts().reindex(all_classes,fill_value=0).to_dict()})
    out=Path(output);out.mkdir(parents=True,exist_ok=True)
    (out/'split_manifest.json').write_text(json.dumps({'protocol':p['protocol'],'train_records':sorted(set(train.record_id.astype(str))),'test_records':sorted(set(test.record_id.astype(str))),'train_patients':sorted(set(groups)),'test_patients':sorted(set(test.patient_id.astype(str))),'cv':'StratifiedGroupKFold','n_splits':n_splits,'seed':seed,'fold_class_counts':fold_class_counts,'ds1_class_counts':y.value_counts().to_dict(),'ds2_class_counts':test.target.astype(str).value_counts().to_dict(),'deviation':p['split_deviation']},indent=2))
    metrics={'features':features,'cv':{'splitter':'StratifiedGroupKFold','n_splits':n_splits},'models':{}}
    for name,(pipe,grid) in _models(seed,quick).items():
        search=GridSearchCV(pipe,grid,scoring='f1_macro',cv=splits,n_jobs=2,refit=True,error_score='raise',return_train_score=False)
        search.fit(train[features],y)
        pred=search.predict(test[features]); labels=['N','S','V','F','Q']
        metrics['models'][name]={'best_cv_macro_f1':float(search.best_score_),'best_params':search.best_params_,'test_macro_f1':float(f1_score(test.target.astype(str),pred,average='macro',labels=labels,zero_division=0)),'test_report':classification_report(test.target.astype(str),pred,labels=labels,output_dict=True,zero_division=0),'confusion_matrix':confusion_matrix(test.target.astype(str),pred,labels=labels).tolist()}
        joblib.dump(search.best_estimator_,out/f'{name}.joblib')
    (out/'metrics.json').write_text(json.dumps(metrics,indent=2))
    return metrics

In [ ]:
RUN_TRAINING = False
if RUN_TRAINING:
    metrics = run_training(FEATURE_PATH, OUTPUT_DIR, seed=42, quick=True)
    print(json.dumps(metrics, indent=2))
else:
    metrics_file = OUTPUT_DIR / "metrics.json"
    if metrics_file.exists():
        metrics = json.loads(metrics_file.read_text())
        print("Loaded existing quick-run metrics from", metrics_file)
    else:
        metrics = None
        print("No saved quick-run metrics found. Set RUN_TRAINING=True to train the models.")

## 10. Evaluate and interpret the current quick run

The metrics below come from the saved quick-run output when available. Accuracy is the fraction of all beats predicted correctly. Precision asks “when the model predicts this class, how often is it right?” Recall asks “of the true beats in this class, how many did it find?” F1 balances precision and recall. Macro-F1 averages class F1 values equally.

In [ ]:
if metrics is not None:
    for model_name, result in metrics["models"].items():
        print(f"\n{model_name}")
        print("  grouped-CV macro-F1:", round(result["best_cv_macro_f1"], 3))
        print("  DS2 macro-F1:", round(result["test_macro_f1"], 3))
        report = pd.DataFrame(result["test_report"]).T
        display(report.loc[["N", "S", "V", "F", "Q"], ["precision", "recall", "f1-score", "support"]].round(3))

In [ ]:
# Plot confusion matrix for one selected model. Rows are true classes; columns are predictions.
if metrics is not None:
    model_name = "random_forest"
    cm = np.asarray(metrics["models"][model_name]["confusion_matrix"])
    labels = ["N", "S", "V", "F", "Q"]
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(labels)), labels)
    ax.set_yticks(range(len(labels)), labels)
    ax.set_xlabel("Predicted class")
    ax.set_ylabel("True class")
    ax.set_title(f"{model_name.replace('_', ' ').title()} — DS2 confusion matrix")
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, f"{cm[i, j]:,}", ha="center", va="center", fontsize=8)
    fig.colorbar(im, ax=ax, label="Number of beats")
    plt.tight_layout()
    plt.show()

### Reading the current result honestly

The current quick run has high overall accuracy but much lower macro-F1. Most test beats are N, so a model can achieve high accuracy while missing rare S or F beats. There are only 7 Q beats in DS2, which is far too few for a stable estimate of Q performance. The right conclusion is not simply “the model is accurate”; report class-wise precision, recall, F1, support counts, and the patient-independent split.

The current feature set also has limitations: several morphology measurements are missing and the useful inputs are mostly RR timing and amplitudes. Better scores would require validated waveform morphology features and enough examples of rare classes—not merely a larger parameter search.

### Saved quick-run summary

| Model | DS2 accuracy | DS2 macro-F1 |
|---|---:|---:|
| Logistic Regression | 93.8% | 0.395 |
| Random Forest | 85.9% | 0.386 |

These values are from the quick run already saved in this project. They are included for context; the notebook's metric cells load the detailed reports from `outputs/smoke/metrics.json` when that file is available. The high accuracy does not mean rare classes are handled well.

## 11. Outputs and reproducibility

The feature builder writes `features.csv`, a feature schema, a data dictionary, and `heartbeat_windows.npz`. The training script writes fitted model files, `metrics.json`, and `split_manifest.json`. The manifest records which records and patients were used in each partition. Keep the protocol file and manifest with reported results.

In [ ]:
print("Feature CSV:", FEATURE_PATH)
print("Protocol:", PROTOCOL_PATH)
print("Output folder:", OUTPUT_DIR)
if (OUTPUT_DIR / "split_manifest.json").exists():
    print("Saved split manifest:", OUTPUT_DIR / "split_manifest.json")

## 12. Re-running from the terminal

From the project root:

```bash
# Build features from local MIT-BIH files
python preprocessing.py --data-dir mitdb --output data/processed/features.csv

# Quick grouped-CV search and DS2 evaluation
python modeling.py --features data/processed/features.csv --output-dir outputs/smoke --quick
```

The notebook and scripts require the project files plus the local dataset. If sharing only the notebook, the professor will also need the MIT-BIH data and dependencies or an exported copy of the processed feature CSV. This is a research/educational classifier, not a medical diagnostic tool.